# MCP (Model Context Protocol) – Databricks

Test connection to a Databricks MCP server and list/call tools. Uses `DatabricksMCPClient` with your workspace client; set the server URL to your UC function schema (e.g. `system/ai` or `catalog/schema`).

**References:** [MCP for beginners](https://github.com/microsoft/mcp-for-beginners) · [MCP workshop](https://github.com/softchris/mcp-workshop) · [MCP concepts](https://softchris.github.io/mcp-workshop/docs/category/mcp-concepts)

In [1]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

In [2]:
import nest_asyncio
nest_asyncio.apply()

In [3]:
workspace_client = WorkspaceClient()
workspace_hostname = workspace_client.config.host
mcp_server_url = f"{workspace_hostname}/api/2.0/mcp/functions/system/ai"

In [4]:
def test_connect_to_server():
    mcp_client = DatabricksMCPClient(server_url=mcp_server_url, workspace_client=workspace_client)
    tools = mcp_client.list_tools()

    print(
        f"Discovered tools {[t.name for t in tools]} "
        f"from MCP server {mcp_server_url}"
    )

    result = mcp_client.call_tool(
        "system__ai__python_exec", {"code": "print('Hello, world!')"}
    )
    print(
        f"Called system__ai__python_exec tool and got result "
        f"{result.content}"
    )


In [5]:
test_connect_to_server()

Discovered tools ['system__ai__python_exec'] from MCP server https://adb-984752964297111.11.azuredatabricks.net/api/2.0/mcp/functions/system/ai
Called system__ai__python_exec tool and got result [TextContent(type='text', text='{"is_truncated":false,"columns":["output"],"rows":[["Hello, world!\\n"]]}', annotations=None, meta=None)]


In [15]:
def test_uc_function():
    mcp_server_url_uc = f"{workspace_hostname}/api/2.0/mcp/functions/shm/default"

    mcp_client = DatabricksMCPClient(
        server_url=mcp_server_url_uc,
        workspace_client=workspace_client
    )
    
    # No await needed
    result = mcp_client.call_tool(
        "shm__default__add_numbers",
        {"x":2, "y": 3}
    )
    
    print(result.content)

In [16]:
test_uc_function()

[TextContent(type='text', text='{"is_truncated":false,"columns":["output"],"rows":[[5]]}', annotations=None, meta=None)]
